In [ ]:
import pandas as pd

url = "http://data.gdeltproject.org/gdeltv2/masterfilelist.txt"
files = pd.read_csv(url, sep=" ", header=None)

# keep only GKG files
gkg_files = files[files[2].str.contains("gkg.csv.zip", na=False)]

/tmp/ipykernel_9533/3487167741.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  files = pd.read_csv(url, sep=" ", header=None)


In [ ]:
# gkg_files = gkg_files[
#     gkg_files[2].str.contains("2019|2020|2021|2022", na=False)
# ]

# urls = gkg_files[2].tolist()

# print("Total files:", len(urls))

gkg_files["timestamp"] = gkg_files[2].str.extract(r'(\d{14})')

# Extract year
gkg_files["year"] = gkg_files["timestamp"].str[:4]

# Select years you want
gkg_files = gkg_files[gkg_files["year"].isin(["2021","2022","2023","2024"])]

/tmp/ipykernel_9533/1402682666.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gkg_files["timestamp"] = gkg_files[2].str.extract(r'(\d{14})')
/tmp/ipykernel_9533/1402682666.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gkg_files["year"] = gkg_files["timestamp"].str[:4]


In [ ]:
print(len(gkg_files))

139474


In [ ]:
gkg_files["hour"] = gkg_files["timestamp"].str[8:10]

sample_hours = ["00","06","12","18"]

sampled_files = gkg_files[gkg_files["hour"].isin(sample_hours)]

In [ ]:
sampled_files = sampled_files.reset_index(drop=True)

In [ ]:
print(len(sampled_files))

23222


In [ ]:
# output_file = "gdelt_finance_news.csv"

# import os
# if not os.path.exists(output_file):
#     open(output_file, "w").close()
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.makedirs("/content/drive/MyDrive/gdelt", exist_ok=True)

In [ ]:
output_file = "/content/drive/MyDrive/gdelt/gdelt_finance_news.csv"

In [ ]:
import requests, zipfile, io, os
import pandas as pd
from tqdm import tqdm

batch_size = 500   # files per batch

for start in range(0, len(sampled_files), batch_size):

    batch = sampled_files.iloc[start:start+batch_size]

    # iterate over URL column
    for url in tqdm(batch[2]):

        try:
            r = requests.get(url, timeout=10)

            z = zipfile.ZipFile(io.BytesIO(r.content))

            df = pd.read_csv(
                z.open(z.namelist()[0]),
                sep="\t",
                header=None,
                usecols=[1,4,8,10,14,15]
            )

            df.columns = [
                "DATE",
                "DocumentIdentifier",
                "V2Themes",
                "V2Locations",
                "V2Organizations",
                "V2Tone"
            ]

            # financial news filter
            df = df[
                df["V2Themes"].str.contains(
                    "ECON_|CRISISLEX_|WB_1104",
                    na=False
                )
            ]

            if len(df) > 0:

                df.to_csv(
                    output_file,
                    mode="a",
                    header=not os.path.exists(output_file),
                    index=False
                )

        except Exception as e:
            print("Error downloading:", url)

    print("Finished batch:", start)

  1%|▏         | 7/500 [00:03<02:54,  2.83it/s]

Error downloading: http://data.gdeltproject.org/gdeltv2/20210101063000.gkg.csv.zip


  5%|▌         | 25/500 [00:07<01:50,  4.28it/s]

Error downloading: http://data.gdeltproject.org/gdeltv2/20210102120000.gkg.csv.zip


  6%|▌         | 28/500 [00:08<01:55,  4.09it/s]

...

Error downloading: http://data.gdeltproject.org/gdeltv2/20231215183000.gkg.csv.zip


 44%|████▍     | 221/500 [01:40<01:59,  2.34it/s]

Error downloading: http://data.gdeltproject.org/gdeltv2/20231222181500.gkg.csv.zip


 44%|████▍     | 221/500 [01:41<02:07,  2.19it/s]


KeyboardInterrupt: 